In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, IntSlider, FloatSlider, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Demonstrate how cascading elementary low-pass sections produces a higher-order filter.</div>
<div><b>Construction:</b> N identical first-order low-pass sections are connected in cascade.</div>
<div><b>Normalization:</b> The pole frequency of each section is adjusted so that the composite filter always has its −3 dB point at ωc.</div>
<div><b>What we see:</b> Increasing the filter order increases the stopband attenuation rate and modifies the phase and group delay.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

order_slider = IntSlider(min=1, max=10, step=1, value=1, description='Order N:', continuous_update=True, style={'description_width':'70px'}, layout=Layout(width='250px'))
wc_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='ωc:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'70px'}, layout=Layout(width='250px'))

# ------------------------------------------------------------
# 3. LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:150px;
    font-size:13px;
    line-height:1.7;
    background:white;
">
<div><span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>Composite filter</div>
<div><span style="display:inline-block; width:32px; border-top:2px dashed gray; vertical-align:middle; margin-right:7px;"></span>Single section</div>
<div><span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>ωc</div>
</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Filter Parameters:</div>")

# ------------------------------------------------------------
# 4. OUTPUT AREAS
# ------------------------------------------------------------

magnitude_output = Output(layout=Layout(width='100%', overflow='hidden'))
phase_output = Output(layout=Layout(width='100%', overflow='hidden'))
delay_output = Output(layout=Layout(width='100%', overflow='hidden'))
info_output = Output(layout=Layout(width='265px', overflow='hidden'))

# ------------------------------------------------------------
# 5. UPDATE FUNCTION
# ------------------------------------------------------------

def update_higher_order_filter(N, wc):

    # --------------------------------------------------------
    # NORMALIZED SECTION POLE
    # --------------------------------------------------------

    wp = wc / np.sqrt(2.0**(1.0 / N) - 1.0)

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.logspace(np.log10(wc / 20.0), np.log10(wc * 100.0), 4001)

    # --------------------------------------------------------
    # SINGLE FIRST-ORDER SECTION
    # --------------------------------------------------------

    H1 = 1.0 / (1.0 + 1j * omega / wp)

    # --------------------------------------------------------
    # CASCADE OF N IDENTICAL SECTIONS
    # --------------------------------------------------------

    H = H1**N

    magnitude1_db = 20.0 * np.log10(np.maximum(np.abs(H1), 1e-12))
    magnitude_db = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))

    phase1 = np.unwrap(np.angle(H1))
    phase = np.unwrap(np.angle(H))

    phase1_deg = np.rad2deg(phase1)
    phase_deg = np.rad2deg(phase)

    group_delay1 = -np.gradient(phase1, omega)
    group_delay = -np.gradient(phase, omega)

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    with magnitude_output:

        magnitude_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.0))

        ax.semilogx(omega, magnitude1_db, linestyle='--', linewidth=1.5, alpha=0.65)
        ax.semilogx(omega, magnitude_db, 'r-', linewidth=2.2)

        ax.axvline(wc, color='black', linestyle=':', linewidth=1.4)
        ax.axhline(-3.0103, color='gray', linestyle='--', linewidth=1.0)

        ax.plot(wc, -3.0103, 'ko', markersize=4)

        ax.set_xlim(omega[0], omega[-1])
        ax.set_ylim(-100.0, 5.0)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Magnitude (dB)', fontsize=11)

        ax.set_title('Higher-Order Low-Pass Filter — Magnitude Response', fontsize=13, fontweight='bold', pad=7)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        fig.subplots_adjust(left=0.11, right=0.98, bottom=0.20, top=0.86)

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    with phase_output:

        phase_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 2.8))

        ax.semilogx(omega, phase1_deg, linestyle='--', linewidth=1.5, alpha=0.65)
        ax.semilogx(omega, phase_deg, 'r-', linewidth=2.2)

        ax.axvline(wc, color='black', linestyle=':', linewidth=1.4)
        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)

        ax.set_xlim(omega[0], omega[-1])

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Phase (degrees)', fontsize=11)

        ax.set_title('Higher-Order Low-Pass Filter — Phase Response', fontsize=13, fontweight='bold', pad=7)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        fig.subplots_adjust(left=0.11, right=0.98, bottom=0.21, top=0.84)

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------

    with delay_output:

        delay_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 2.8))

        ax.semilogx(omega, group_delay1, linestyle='--', linewidth=1.5, alpha=0.65)
        ax.semilogx(omega, group_delay, 'r-', linewidth=2.2)

        ax.axvline(wc, color='black', linestyle=':', linewidth=1.4)

        ax.set_xlim(omega[0], omega[-1])
        ax.set_ylim(bottom=0.0)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Group Delay (s)', fontsize=11)

        ax.set_title('Higher-Order Low-Pass Filter — Group Delay', fontsize=13, fontweight='bold', pad=7)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        fig.subplots_adjust(left=0.11, right=0.98, bottom=0.21, top=0.84)

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # INFORMATION FRAME
    # --------------------------------------------------------

    slope = -20 * N
    phase_inf = -90 * N

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 9px;
        margin-top:8px;
        font-size:12px;
        line-height:1.8;
        background:white;
        width:255px;
        box-sizing:border-box;
        white-space:nowrap;
    ">

        <div>
            <b>Order N:</b>
            <span style="color:#0066cc;">{N}</span>
        </div>

        <div>
            <b>Composite cutoff ωc:</b>
            <span style="color:#0066cc;">{wc:.2f} rad/s</span>
        </div>

        <div>
            <b>Section pole ωp:</b>
            <span style="color:#0066cc;">{wp:.3f} rad/s</span>
        </div>

        <div>
            <b>|H(jωc)|:</b>
            <span style="color:#0066cc;">−3.01 dB</span>
        </div>

        <div>
            <b>Asymptotic slope:</b>
            <span style="color:#0066cc;">{slope} dB/decade</span>
        </div>

        <div>
            <b>Final phase:</b>
            <span style="color:#0066cc;">{phase_inf}°</span>
        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 6. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(update_higher_order_filter, {'N': order_slider, 'wc': wc_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 7. LEFT COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, order_slider, wc_slider, info_output], layout=Layout(width='285px', min_width='285px', flex='0 0 285px', align_items='flex-start', padding='2px 0px 0px 4px', overflow='hidden'))

# ------------------------------------------------------------
# 8. RIGHT COLUMN
# ------------------------------------------------------------

right_column = VBox([magnitude_output, phase_output, delay_output], layout=Layout(width='auto', min_width='0px', flex='1 1 auto', align_items='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 9. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, right_column], layout=Layout(width='100%', max_width='100%', align_items='flex-start', justify_content='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 10. FINAL DISPLAY
# ------------------------------------------------------------

display(description)
display(main_area)
display(interactive_controls)